In [ ]:
import os
import asyncio
import httpx
from google import genai
from dotenv import load_dotenv

load_dotenv()

TMDB_TOKEN = os.getenv("token")

BASE_URL = "https://api.themoviedb.org/3"

headers = {
    "Authorization": f"Bearer {TMDB_TOKEN}",
    "accept": "application/json"
}
model = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError(".env 파일에 GEMINI_API_KEY를 설정해 주세요.")

client = genai.Client(api_key=api_key)
# 모델 이름은 바뀔 수 있으므로 환경 변수로 교체할 수 있게 둡니다.
model = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")
print("준비 완료 / 사용 모델:", model)

In [ ]:
async def get_movie_list(list_type, page_count=2):
    url = f"{BASE_URL}/movie/{list_type}"

    if page_count < 2:
        page_count = 2
    async with httpx.AsyncClient() as http_client:
        pages = await asyncio.gather(
          *(
            http_client.get(
                url,
                headers=headers,
                params={"language":"ko-KR", "page":page}
            ) 
            for page in range(1, page_count + 1))
        )

    movies = []
    for response in pages:
        data = response.json()
        movies.extend(data["results"])
    return movies

        
async def get_movie_detail(movie_id):
    url = f"{BASE_URL}/movie/{movie_id}"
    async with httpx.AsyncClient() as http_client:
        response = await http_client.get(
            url,
            headers=headers,
            params={"language":"ko-KR"}
        )
    data = response.json()
    return data

# movies = await get_movie_list("popular", 2)
# movie_id = movies[0]["id"]
# detail = await get_movie_detail(movie_id)
# print("제목:", detail["title"])
# print("개봉일:", detail["release_date"])
# print("평점:", detail["vote_average"])
# print("줄거리:", detail["overview"])


async def search_movies(query, page=1):
    url = f"{BASE_URL}/search/movie"
    async with httpx.AsyncClient() as http_client:
        response = await http_client.get(
            url,
            headers=headers,
            params={
                "query":query,
                "language":"ko-KR",
                "page":page
            }
        )
    data = response.json()
    return data["results"]

# search_results = await search_movies("인셉션")
# print("검색 결과 수:", len(search_results))

# for movie in search_results:
#     print(
#         movie["title"],
#         movie["release_date"],
#         movie["vote_average"]
#     )


async def get_top_rated_movies(list_type, page_count=2, result_count=10, min_vote_count=100):
    movies = await get_movie_list(list_type, page_count)
    movies = [
        movie
        for movie in movies
        if movie["vote_count"] >= min_vote_count
    ]

    movies.sort(key=lambda movie : (movie["vote_average"], movie["vote_count"]), reverse=True)
    return movies[:result_count]

# top_movies = await get_top_rated_movies(
#     "popular",
#     page_count=2,
#     result_count=5
# )

# for movie in top_movies:
#     print(
#         "제목:", movie["title"],
#         "| 평점:", movie["vote_average"],
#         "| 평가 수:", movie["vote_count"]
#     )

get_movie_list_tool = {
    "type": "function",
    "name": "get_movie_list",
    "description": "인기, 현재 상영, 평점순, 개봉 예정 등의 영화 목록을 최소 2페이지 이상 조회합니다.",
    "parameters": {
        "type": "object",
        "properties": {
            "list_type": {
                "type": "string",
                "description": "조회할 영화 목록 종류",
                "enum": [
                    "popular",
                    "now_playing",
                    "top_rated",
                    "upcoming"
                ]
            },
            "page_count": {
                "type": "integer",
                "description": "조회할 페이지 수. 최소 2페이지 이상 조회합니다.",
                "minimum": 2
            }
        },
        "required": ["list_type"]
    }
}


get_movie_detail_tool = {
    "type": "function",
    "name": "get_movie_detail",
    "description": "목록이나 검색 결과에서 얻은 ID로 영화의 상세 정보를 조회한다.",
    "parameters": {
        "type": "object",
        "properties": {
            "movie_id": {
                "type": "integer",
                "description": "조회한 영화의 ID",
            },
        },
        "required": ["movie_id"]
    }
}

search_movies_tool = {
    "type": "function",
    "name": "search_movies",
    "description": "제목 등의 검색어로 영화를 찾는다.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "검색어",
            },
            "page": {
                "type": "integer",
                "description": "조회할 페이지 수.",
            }
        },
        "required": ["query", "page"]
    }
}


get_top_rated_movies_tool = {
    "type": "function",
    "name": "get_top_rated_movies",
    "description": "여러 페이지를 조회한 뒤 평점과 평가 수를 기준으로 상위 영화를 선별한다.",
    "parameters": {
        "type": "object",
        "properties": {
            "list_type": {
                "type": "string",
                "description": "조회할 영화 목록 종류",
                "enum": [
                    "popular",
                    "now_playing",
                    "top_rated",
                    "upcoming"
                ]
            },
            "page_count": {
                "type": "integer",
                "description": "조회할 페이지 수. 최소 2페이지 이상 조회합니다.",
                "minimum": 2
            },
            "result_count": {
                "type": "integer",
                "description": "결과 수.",
            },
            "min_vote_count": {
                "type": "integer",
                "description": "최소 평가 수",
            }
        },
        "required": ["list_type", "page_count", "result_count", "min_vote_count"]
    }
}


TOOL_FUNCTIONS = {
    "get_movie_list": get_movie_list,
    "get_movie_detail": get_movie_detail,
    "search_movies": search_movies,
    "get_top_rated_movies": get_top_rated_movies,
}

tools = [
    get_movie_list_tool, 
    get_movie_detail_tool,
    search_movies_tool,
    get_top_rated_movies_tool
]

In [ ]:
import json

async def execute_tool_call(step):
    tool_result = await TOOL_FUNCTIONSstep.name

    return {
        "type": "function_result",
        "name": step.name,
        "call_id": step.id,
        "result": [
            {
                "type": "text",
                "text": json.dumps(
                    tool_result,
                    ensure_ascii=False
                )
            }
        ],
    }

def create_interaction(input_data, previous_interaction_id):
    return client.interactions.create(
        model=model,
        input=input_data,
        previous_interaction_id=previous_interaction_id,
        tools=tools,
        store=True,
    )

async def movie_agent():
    previous_interaction_id = None

    while True:
        user_input = input("사용자 : ")
        
        if user_input == "종료":
            break

        interaction = create_interaction(user_input, previous_interaction_id)
        previous_interaction_id = interaction.id

        while True:
            function_results = []

            for step in interaction.steps:
                if step.type == "function_call":
                    function_result = await execute_tool_call(step)
                    function_results.append(function_result)

            if function_results:
                interaction = create_interaction(function_results, interaction.id)
                previous_interaction_id = interaction.id
            else:
                print(interaction.output_text)
                break
        print()
await movie_agent()
